# Part 4: More advanced networks

__Before starting, we recommend you enable GPU acceleration if you're running on Colab.__

In [ ]:
# Install Torchbearer if it is not already available
try:
    import torchbearer
except ImportError:
    %pip install torchbearer
    import torchbearer

## Branching and merging

Recent network models, such as ResNet and GoogLeNet, do not follow a single straight path from input to output. Instead, these models incorporate branches and merges to create a computation graph. Branching and merging is easy to implement in PyTorch, as shown in the following example.

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import MNIST
from torchbearer import Trial

# Load the data here so that this notebook can be run independently.
transform = transforms.ToTensor()
full_trainset = MNIST(".", train=True, download=True, transform=transform)
testset = MNIST(".", train=False, download=True, transform=transform)

# Use a validation set for decisions made while developing the models, and
# reserve the test set for a final evaluation.
generator = torch.Generator().manual_seed(7)
trainset, valset = random_split(full_trainset, [50000, 10000], generator=generator)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True)
valloader = DataLoader(valset, batch_size=128, shuffle=False)
testloader = DataLoader(testset, batch_size=128, shuffle=False)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
class BranchModel(nn.Module):
    def __init__(self):
        super(BranchModel, self).__init__()
        self.left  = nn.Conv2d(1, 16, (1, 1), padding=0)
        self.right = nn.Conv2d(1, 16, (5, 5), padding=2)
        self.fc1 = nn.Linear(16*14*14, 128)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        out_l = self.left(x)
        out_l = F.relu(out_l)

        out_r = self.right(x)
        out_r = F.relu(out_r)

        out = out_l + out_r
        
        out = F.max_pool2d(out, (2,2))
        out = F.dropout(out, p=0.2, training=self.training)
        out = out.view(out.shape[0], -1)
        out = self.fc1(out)
        out = F.relu(out)
        out = self.fc2(out)

        return out

This defines a variant of our initial simple CNN model in which the input is split into two paths and then merged again; the left hand path consists of a 1x1 convolution layer, whilst the right-hand path has a 5x5 convolutional layer. The 1x1 convolutions will have the effect of increasing the number of bands in the input from 1 to 16 (with each band a (potentially different) scalar multiple of the input). Padding is used to ensure the feature maps have the same shape on the left and right branches. In this case the left and right branches are merged by summing them together (element-wise, layer by layer).

__Use the code block below to train `BranchModel` and record its validation performance.__ Keep the resulting trial so that you can compare it with the residual model. Do not evaluate on the test set yet.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

## Residual connections

The two branches in `BranchModel` both learn transformations of the input. A residual block uses a particularly important kind of branch: an **identity shortcut**. If the learned branch computes $F(x)$, the block produces

$$y = F(x) + x.$$

The shortcut passes the input to the addition unchanged. This provides a direct path through the network for both activations and gradients, while the learned branch only has to model a residual change to its input.

Element-wise addition requires $F(x)$ and $x$ to have exactly the same shape. The block below therefore preserves both the number of channels and the spatial dimensions.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        # Add the identity shortcut, then apply the final activation.
        # YOUR CODE HERE
        raise NotImplementedError()

        return out

__Complete the residual block above, then run the following shape check.__ Notice that the shortcut contains no learned parameters: `identity` is just another reference to the input tensor.

In [ ]:
block = ResidualBlock(16)
example = torch.randn(4, 16, 28, 28)
result = block(example)

assert result.shape == example.shape
print("input shape: ", example.shape)
print("output shape:", result.shape)

We can now use the block as an ordinary PyTorch module. This small classifier first maps the single-channel MNIST input to 16 channels, processes it through two residual blocks, then uses global average pooling to produce one value per channel before classification.

In [ ]:
class ResidualCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),
        )
        self.residual_blocks = nn.Sequential(
            ResidualBlock(16),
            ResidualBlock(16),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(16, 10)

    def forward(self, x):
        out = self.stem(x)
        out = self.residual_blocks(out)
        out = self.pool(out)
        out = out.flatten(1)
        return self.classifier(out)

__Train `ResidualCNN` with Torchbearer and compare its validation performance with `BranchModel`.__ Select the model using validation performance, then evaluate the selected trial once on the held-out test set and report that final accuracy.

Then answer these questions:

1. Why can the identity and learned paths in `ResidualBlock` be added directly?
2. Suppose the learned path changes from 16 to 32 channels and halves the spatial dimensions using stride 2. What must happen to the shortcut before the paths can be added? Hint: consider a $1\times1$ convolution with stride 2.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

## Going further

None of the network topologies we have experimented with thus far are optimised. Nor are they reproductions of network topologies from recent papers.

__There is a lot of opportunity for you to tune and improve upon these models. What is the best error rate score you can achieve?__
